In [1]:
from pyspark.sql import functions as F

df_sol = spark.sql(
    """
    SELECT * FROM silver.fato_solicitacoes 
    WHERE municipio = 'Osasco' 
    AND servico = 'Atendimento ao trabalhador'
    """
)

df_campos = spark.sql(
    """
    SELECT * FROM silver.fato_campos 
    WHERE municipio = 'Osasco' 
    AND servico = 'Atendimento ao trabalhador'
    """
)

df_campos_pivot = (
    df_campos
    .groupBy("id_os")
    .pivot("campo")
    .agg(F.first("valor"))
)

df = df_sol.join(df_campos_pivot, on="id_os", how="left")

colunas_demanda = [c for c in df.columns if c.startswith("demanda_")]
for c in colunas_demanda:
    df = df.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c))
        .cast("int")
    )
df = df.fillna(0, subset=colunas_demanda)

df = df.withColumn(
    "tempo_atendimento_minutos",
    (F.col("data_finalizacao").cast("long") - F.col("data_criacao").cast("long")) / 60
)

(
    df
    .write.mode("overwrite")
    .format("delta")
    .saveAsTable("gold.osasco_atendimento_trabalhador")
)

StatementMeta(, 8df23e25-68fa-41dc-843b-4bc4d9a82c02, 3, Finished, Available, Finished, False)